- Entradas: "PwmD", "PwmE", "sPwm", "dPwm"
- Saida: Theta
- Loss = L_d + L_p 


In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib
import os
TITLES = ["Train_1", "Train_2", "Test_1", "Test_2", "Test_3", "Val", "LSG_1", "LSG_2"]

PREDICTORS = ["PwmD", "PwmE", "sPwm", "dPwm"]   
TARGET_INT = ["Theta"]  
TARGET = ["DeltaTheta"]
     
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   
     
TIME_STEPS = 7
TS = 0.07

In [20]:
Datasets = []
for i in range(4):
    Dataset = pd.read_excel(f"../../00-RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(4):   
    Dataset = pd.read_csv(f"../../00-Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS
    
    Dataset["sPwm"] = Dataset["PwmD"] + Dataset["PwmE"]
    Dataset["dPwm"] = Dataset["PwmD"] - Dataset["PwmE"]

    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [21]:

NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)

# concatena
Train = pd.concat([Train1, Train2], ignore_index=True)

for i in range(6):
    CurrentTestDataset = Datasets[i + 2].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

In [22]:
os.makedirs("./scalers", exist_ok=True)
os.makedirs("./Data", exist_ok=True)

with pd.ExcelWriter("./Data/NormDatasets.xlsx", engine="openpyxl") as writer_norm:
    for title, normDataset in zip(TITLES, NormDatasets):
        normDataset.to_excel(writer_norm, sheet_name=title[:31], index=False)

with pd.ExcelWriter("./Data/Datasets.xlsx", engine="openpyxl") as writer:
    for title, Dataset in zip(TITLES, Datasets):
        Dataset.to_excel(writer, sheet_name=title[:31], index=False)
        
joblib.dump(SCALER, "./scalers/scaler.pkl")
joblib.dump(OUT_SCALER, "./scalers/out_scaler.pkl")

mean_tf = tf.constant(OUT_SCALER.mean_[0], dtype=tf.float32)
std_tf  = tf.constant(OUT_SCALER.scale_[0], dtype=tf.float32)        

In [23]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(y_val)}")

Dimensão da entrada: (1945, 7, 4)
Dimensão da saida: (1945, 1)
Dimensão da entrada: (1264, 7, 4)
Dimensão da saida: (1264, 1)


In [24]:
Wd_train =  Train["Wd"].values[:len(x_train)]
We_train =  Train["We"].values[:len(x_train)]
theta_train = Train["Theta"].values[:len(x_train)]

print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada fisica : {np.shape(Wd_train)}")
print(f"Dimensão da entrada fisica: {np.shape(We_train)}")

Dimensão da entrada: (1945, 7, 4)
Dimensão da saida: (1945, 1)
Dimensão da entrada fisica : (1945,)
Dimensão da entrada fisica: (1945,)


$$ \dot{\theta} = \frac{R}{2L} (\phi_d - \phi_e) $$
$$ \dot{x} = \frac{R}{2} (\phi_d + \phi_e) (\cos(\theta))$$
$$ \dot{y} = \frac{R}{2} (\phi_d + \phi_e) (\sin(\theta)) $$

In [ ]:
R = tf.constant(0.0328, dtype=tf.float32)
L = tf.constant(0.0615, dtype=tf.float32)
dt = tf.constant(TS, dtype=tf.float32)

def CinematicModel(Wd, We, theta):

    dtheta_cin = (R / (2 * L)) * (Wd - We)
    return [dtheta_cin]
    

In [26]:

def NumericalIntegration(dataset, dq):

    q = [None] * OUTPUT_SIZE

    init_vals = np.array([
        dataset[name].iloc[0] for name in TARGET_INT
    ])

    for j in range(OUTPUT_SIZE):
        q[j] = init_vals[j] + np.cumsum(dq[j] * TS)

    return q

def GetCin(dataset): 
    dq = CinematicModel(tf.convert_to_tensor(dataset["Wd"].values, dtype=tf.float32),
                        tf.convert_to_tensor(dataset["We"].values, dtype=tf.float32), 
                        tf.convert_to_tensor(dataset["Theta"].values, dtype=tf.float32))
    q = NumericalIntegration(dataset, dq)
    return np.vstack(q).T

In [28]:
def BuildRNN(architecture, initializer, regularizer):

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

    for i, units in enumerate(architecture):

        return_sequences = (i < len(architecture) - 1)

        model.add(
            tf.keras.layers.SimpleRNN(
                units,
                activation='tanh',
                return_sequences=return_sequences,
                kernel_initializer=initializer,
                kernel_regularizer=regularizer,
                recurrent_regularizer=regularizer,
                bias_regularizer=regularizer
            )
        )

    model.add(
        tf.keras.layers.Dense(
            OUTPUT_SIZE,
            activation="linear",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer,
            bias_regularizer=regularizer
        )
    )

    return model

In [29]:
@tf.function
def train_step(model, optimizer, x, dy, Wd, We, theta0, Ld, Lp):

    with tf.GradientTape() as tape:

        dy_pred = model(x, training=True)
        
        # loss dos dados
        data_loss = tf.reduce_mean(tf.square(dy_pred - dy))
        
        # termo físico
        physics = tf.stack(CinematicModel(Wd, We, theta0), axis=1)   
             
        # normalização correta
        physics_norm = (physics - mean_tf) / std_tf

        physics_loss = tf.reduce_mean(tf.square(dy_pred - physics_norm))

        loss = Ld * data_loss +  Lp * physics_loss 

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

In [30]:
Wd_train = tf.convert_to_tensor(Wd_train, dtype=tf.float32)
We_train = tf.convert_to_tensor(We_train, dtype=tf.float32)
theta_train = tf.convert_to_tensor(theta_train, dtype=tf.float32)

x_train_tf = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.float32)

x_val_tf = tf.convert_to_tensor(x_val, dtype=tf.float32)
y_val_tf = tf.convert_to_tensor(y_val, dtype=tf.float32)

In [31]:
def EarlyStopping(model, best_loss, counter, best_weights, min_delta=1e-3):
    val_pred = model(x_val_tf, training=False)
    val_loss = tf.reduce_mean(tf.square(val_pred - y_val_tf))
    
    if val_loss < (best_loss - min_delta):
        best_loss = val_loss
        counter = 0
        best_weights = model.get_weights()

    else:
        counter += 1

    return best_loss, counter, best_weights, val_loss

def TrainPINN(model, optimizer, Ld, Lp, patience=200, best_loss=np.inf):
    counter = 0
    best_weights = model.get_weights()

    for epoch in range(20000):

        loss = train_step(model, optimizer, x_train_tf, y_train_tf,
                          Wd_train, We_train, theta_train, Ld, Lp)
        
        best_loss, counter, best_weights, val_loss =  EarlyStopping(model, best_loss, counter, best_weights)
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            model.set_weights(best_weights)
            break

        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Train Loss {loss.numpy():.6f} | Val Loss {val_loss.numpy():.6f}")

In [32]:
def PlotOut(ax, title, target_name, y_true, y_pred, y_cin):
    time = (np.arange(0, len(y_pred), 1).astype(float) * 0.07).round(5)

    ax.plot(time, y_true, linestyle='-', linewidth=1.5, color='tab:blue', label='Amostras Reais')
    ax.plot(time, y_pred, linestyle='--', linewidth=1.5, color='tab:orange', label='Valores preditos')
    ax.plot(time, y_cin, linestyle=':', linewidth=2, color='tab:green', label='Modelo Cinemático')

    ax.set_title(f'{title} - {target_name}')
    ax.set_xlabel('Tempo [s]')
    ax.set_ylabel(target_name)
    ax.legend()
    ax.grid(True)


def EvalModel(model):
    # n_datasets = len(Datasets)
    n_targets = len(TARGET)
    # fig, axs = plt.subplots(n_datasets, n_targets, figsize=(6 * n_targets, 4 * n_datasets))
    

    # estrutura das métricas
    metrics = {
        name: {
            "R2_Train_1": [], "R2_Train_2": [],
            "R2_Val": [],
            "R2_Test_1": [], "R2_Test_2": [], "R2_Test_3": [],
            "R2_LSG_1": [], "R2_LSG_2": [],
            "MSE_Train_1": [], "MSE_Train_2": [],
            "MSE_Val": [],
            "MSE_Test_1": [], "MSE_Test_2": [], "MSE_Test_3": [],
            "MSE_LSG_1": [], "MSE_LSG_2": [],

        }
        for name in TARGET_INT
    }

    for i, NormDataset in enumerate(NormDatasets):

        x = NormDataset[PREDICTORS] # Normalizado
        y = Datasets[i][TARGET_INT] # Real

        x, y = CreateSequences(x, y, TIME_STEPS)

        # predição da rede
        pred = model(tf.convert_to_tensor(x, dtype=tf.float32)).numpy()

        # desnormalização
        y_pred_diff = OUT_SCALER.inverse_transform(pred)

        # arrays reconstruídos
        y_true = y
        y_pred = np.zeros_like(y_pred_diff)
        y_cin = GetCin(Datasets[i])
        
        y_cin = y_cin[:y_true.shape[0]]

        # valores iniciais reais
        init_vals = np.array([
            Datasets[i][name].iloc[0] for name in TARGET_INT
        ])

        # reconstrução por integração cumulativa
        for j in range(n_targets):
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred_diff[:, j] * TS)

        # cálculo das métricas
        for j, name in enumerate(TARGET_INT):

            r2 = r2_score(y_true[:, j], y_pred[:, j])
            mse = mean_squared_error(y_true[:, j], y_pred[:, j])

            metrics[name][f"R2_{TITLES[i]}"].append(r2)
            metrics[name][f"MSE_{TITLES[i]}"].append(mse)

            print(f"{name} | {TITLES[i]} -> " f"R² = {r2:.4f}, MSE = {mse:.4e}")

            # # Seleciona o eixo correto (funciona mesmo com 1x1, 1x2 ou 3x2)
            # ax = axs[i][j] if n_datasets > 1 and n_targets > 1 else (
            #     axs[j] if n_targets > 1 else axs[i] if n_datasets > 1 else axs
            # )
            # PlotOut(ax, TITLES[i], name, y_true[:, j], y_pred[:, j], y_cin)

    # plt.tight_layout()

    return metrics

In [33]:
def to_scalar(x):
    return float(x[0]) if isinstance(x, list) else float(x)

In [34]:
def UpdateRow(metrics, arch, Ld, Lp, r, seed, excel_file):

    model_name = f"model_arch{'-'.join(map(str, arch))}_r{r}_seed{seed}"

    row = {
        "model": model_name,
        "Neurons": arch,
        "Ld": Ld,
        "Lp": Lp,
        "reg": r,
        "seed": seed,
    }

    for name in TARGET_INT:
        row.update({
            f"R2_Train_1_{name}": to_scalar(metrics[name]["R2_Train_1"]),
            f"MSE_Train_1_{name}": to_scalar(metrics[name]["MSE_Train_1"]),
            f"R2_Train_2_{name}": to_scalar(metrics[name]["R2_Train_2"]),
            f"MSE_Train_2_{name}": to_scalar(metrics[name]["MSE_Train_2"]),
            f"R2_Val_{name}": to_scalar(metrics[name]["R2_Val"]),
            f"MSE_Val_{name}": to_scalar(metrics[name]["MSE_Val"]),
            f"R2_Test_1_{name}": to_scalar(metrics[name]["R2_Test_1"]),
            f"MSE_Test_1_{name}": to_scalar(metrics[name]["MSE_Test_1"]),
            f"R2_Test_2_{name}": to_scalar(metrics[name]["R2_Test_2"]),
            f"MSE_Test_2_{name}": to_scalar(metrics[name]["MSE_Test_2"]),
            f"R2_Test_3_{name}": to_scalar(metrics[name]["R2_Test_3"]),
            f"MSE_Test_3_{name}": to_scalar(metrics[name]["MSE_Test_3"]),
        })

    df = pd.DataFrame([row])

    try:
        old = pd.read_excel(excel_file)
        new_df = pd.concat([old, df], ignore_index=True)
        new_df.to_excel(excel_file, index=False)
    except FileNotFoundError:
        df.to_excel(excel_file, index=False)

    print(f"Modelo {arch} | Ld={Ld} Lp={Lp} r={r} seed={seed} salvo.")


In [35]:
def ExportModel(model, model_name):

    os.makedirs("weights", exist_ok=True)
    os.makedirs("models", exist_ok=True)

    weights_path = f"weights/{model_name}.weights.h5"
    model_path = f"models/{model_name}.keras"

    model.save_weights(weights_path)
    model.save(model_path)

    print(f"Modelo salvo em:\n{model_path}\n{weights_path}")

In [36]:
from tensorflow.keras.optimizers import Adam
from itertools import product

N_MODELS = 5
seeds = np.random.choice(range(1, 10000), size=N_MODELS, replace=False)

architectures = [[16],
    [32], [64], [128], [264],
    [8, 4], [16, 8], [32, 16], [64, 32], [128, 64], [264, 128],
    [16, 8, 4], [32, 16, 8], [128, 64, 32], [264, 128, 64]
]

Ld_Lp = [[0.3,0.7], [0.7,0.3]]
r_values = [0.01, 0.9]

results = {}
# produto cartesiano de todos hiperparâmetros
for arch, (Ld, Lp), r in product(architectures, Ld_Lp, r_values):

    for i, s in enumerate(seeds):

        tf.keras.backend.clear_session()

        init = initializers.RandomNormal(seed=int(s))
        reg = tf.keras.regularizers.l2(r)
        model = BuildRNN(arch, init, reg)
        model.build((None, TIME_STEPS, INPUT_SIZE))

        opt = Adam(learning_rate=0.001)
        opt.build(model.trainable_variables)

        TrainPINN(
            model,
            Ld=Ld,
            Lp=Lp,
            optimizer=opt
        )
        ExportModel(model, model_name=f"model_arch{'-'.join(map(str, arch))}_r{r}_seed{s}")
        metrics = EvalModel(model)
        UpdateRow(metrics, arch, Ld, Lp, r, s, excel_file="resultados.xlsx")

Epoch 0 | Train Loss 0.996326 | Val Loss 0.818921
Epoch 100 | Train Loss 0.805613 | Val Loss 0.585393
Epoch 200 | Train Loss 0.737645 | Val Loss 0.588557
Epoch 300 | Train Loss 0.683840 | Val Loss 0.565414
Epoch 400 | Train Loss 0.651872 | Val Loss 0.586266
Epoch 500 | Train Loss 0.630960 | Val Loss 0.613595
Early stopping at epoch 506
Modelo salvo em:
models/model_arch16_r0.01_seed8888.keras
weights/model_arch16_r0.01_seed8888.weights.h5
Theta | Train_1 -> R² = 0.6485, MSE = 8.0319e-02
Theta | Train_2 -> R² = 0.8114, MSE = 4.2764e-02
Theta | Test_1 -> R² = 0.6689, MSE = 7.6169e-02
Theta | Test_2 -> R² = 0.8363, MSE = 3.8567e-02
Theta | Test_3 -> R² = -0.8436, MSE = 6.0624e-01
Theta | Val -> R² = -3.9644, MSE = 1.6247e+00
Theta | LSG_1 -> R² = -1.7040, MSE = 1.1108e+00
Theta | LSG_2 -> R² = -3.1648, MSE = 1.4274e+00
Modelo [16] | Ld=0.3 Lp=0.7 r=0.01 seed=8888 salvo.
Epoch 0 | Train Loss 0.981218 | Val Loss 0.813271
Epoch 100 | Train Loss 0.745448 | Val Loss 0.589252
Epoch 200 | Train 